In [8]:
import pandas as pd
from scipy.io import wavfile
import glob
import re
import tqdm
import numpy as np
from scipy.signal import butter, filtfilt, welch, spectrogram
import matplotlib.pyplot as plt
from matplotlib.widgets import Button
from matplotlib.gridspec import GridSpec


experiments = [433]
headmic_channels = [118,35]
no_channels = 10

# Functions to sort file names correctly
def atoi(text):
    return int(text) if text.isdigit() else text

def natural_keys(text):
    return [atoi(c) for c in re.split(r'(\d+)', text)]

In [11]:
for exp in experiments:
    general_path = "D:/big_setup/experiment_{}/concatenated_data_cam_mic_sync/".format(exp)

    # getting the das files
    das_files = glob.glob(f"//sanesstorage.cns.nyu.edu/archive/Niegil/das/training_data/data/exp_{exp}*annotations.csv")
    das_files.sort(key=natural_keys)


    # Getting the channels
    channel_files = {}
    for idx in range(no_channels):
        temp_str = "%02d"%(idx)
        channel_files[idx] = glob.glob(general_path+f"channel_{temp_str}_*.wav")
        channel_files[idx].sort(key=natural_keys)


    length_of_experiment = len(glob.glob(general_path+"*.mp4"))
    powers = []
    powers_ = []

    for idx in tqdm.tqdm(range(17)):
        channels = {}
        # getting the channels
        for i in range(no_channels):
            file = channel_files[i][idx]
            file_name = file.split("\\")[-1]
            if "%03d"%(idx) in file:
                sr,channels[i] = wavfile.read(file)
            else:
                raise Exception(f"Error finding channel file({file}) for idx {idx}")
            
        MAX_AUDIO_LEN = min(len(ch) for ch in channels.values()) / sr

        # getting the das file
        index = [id for id, s in enumerate(das_files) if "%03d"%(idx) in s]
        if len(index) == 1:
            file = das_files[index[0]]
            vox_times = pd.read_csv(file) 
            vox_times["label"] = None
        # pass
        else:
            print(f"Number of DAS files found for index {idx} is {len(index)}")
            raise Exception(f"More than one or less than one of das annotations files found for index {idx}")
        
        

        all_vox = []

        vals = list(zip(
            vox_times[vox_times["name"]=="vox"].index,
            vox_times[vox_times["name"]=="vox"]["start_seconds"],
            vox_times[vox_times["name"]=="vox"]["stop_seconds"]
        ))

        for i in range(len(vals)):
            vox_idx, start, stop = vals[i]

            # # Look ahead safely
            # if i < len(vals) - 1:
            #     _, next_start, _ = vals[i + 1]
            # else:
            #     next_start = None

            # # --- Sync correction ---
            # diff = stop - start

            # if diff < 0.03:
            #     add_amt = 0.03
            # elif diff > 0.15:
            #     add_amt = 0.1
            # else:
            #     add_amt = 0.03 + (diff - 0.03) * (0.1 - 0.03) / (0.15 - 0.03)

            # stop += add_amt / 2
            # start -= add_amt / 2

            # # Prevent overlap with next segment
            # if next_start is not None and stop > next_start:
            #     stop = next_start - 0.005

            # # Clamp to 2-minute max length
            # stop = min(stop, MAX_AUDIO_LEN)

            # # Also ensure start is not negative
            # start = max(start, 0)

            # # Skip if invalid after clamping
            # if stop <= start:
            #     continue

            start_idx = int(sr * start)
            stop_idx = int(sr * stop)

            # Faster channel averaging
            segment = np.array([channels[ch][start_idx:stop_idx] for ch in range(no_channels)])
            avg_audio = np.mean(segment, axis=0)

            all_vox.append(avg_audio)
            all_vox.append(np.zeros(1000))

        try:
            wavfile.write(general_path+f"{idx}_test.wav",rate=sr, data=np.concatenate(all_vox).ravel())
            print("Written")
        except:
            continue




  6%|▌         | 1/17 [00:00<00:05,  2.93it/s]

Written


 12%|█▏        | 2/17 [00:00<00:04,  3.65it/s]

Written


 18%|█▊        | 3/17 [00:00<00:03,  4.02it/s]

...

Written


 94%|█████████▍| 16/17 [00:17<00:02,  2.22s/it]

Written


100%|██████████| 17/17 [00:20<00:00,  1.18s/it]

Written


In [5]:
channels.values()

dict_values([array([ 0.00717163,  0.        ,  0.00411987, ...,  0.00473022,
       -0.00015259,  0.00656128], shape=(14997924,), dtype=float32), array([ 0.00091553, -0.00137329, -0.00198364, ..., -0.00411987,
       -0.00518799, -0.0038147 ], shape=(14997924,), dtype=float32), array([ 0.0012207 , -0.0038147 ,  0.00549316, ...,  0.        ,
        0.00396729,  0.00228882], shape=(14997924,), dtype=float32), array([-0.00015259,  0.00061035,  0.00152588, ...,  0.00808716,
        0.00473022, -0.00350952], shape=(14997924,), dtype=float32), array([-0.00061035,  0.00198364,  0.00152588, ..., -0.00259399,
        0.00732422,  0.0088501 ], shape=(14997924,), dtype=float32), array([-0.00015259,  0.00076294, -0.00259399, ...,  0.00015259,
       -0.00030518,  0.00183105], shape=(14997924,), dtype=float32), array([-0.00152588,  0.00167847, -0.00289917, ...,  0.00244141,
       -0.00045776,  0.00015259], shape=(14997924,), dtype=float32), array([0.00350952, 0.00137329, 0.00518799, ..., 0.008850